# Intro to using Flask with PyTorch

Use this Notebook together with the app.py file that introduces Flask itself. Once you understand

* how Flask works in general (with the hello world app.py file)
* the kinds of transforms/prep we have to do with our model, images, etc
* how to rewrite the app.py file to work with our model

then you will know exactly how to deploy PyTorch models with Flask!

This notebook builds on the PyTorch tutorial available here: https://guyuena.github.io/PyTorch-study-Tutorials/intermediate/flask_rest_api_tutorial.html

## What comes below is, (1) Image and model prep

The next two steps are really just testing out the transformations that we need to do on an image in order to make sure it is a tensor that the model will be able to accept

In [1]:
import io

import torchvision.transforms as transforms
from PIL import Image

def transform_image(image_bytes):
    my_transforms = transforms.Compose([transforms.Resize(255),
                                        transforms.CenterCrop(224),
                                        transforms.ToTensor(),
                                        transforms.Normalize(
                                            [0.485, 0.456, 0.406],
                                            [0.229, 0.224, 0.225])])
    image = Image.open(io.BytesIO(image_bytes))
    return my_transforms(image).unsqueeze(0)

In [2]:
# Test this
with open("kitten.jpg", 'rb') as f:
    image_bytes = f.read()
    tensor = transform_image(image_bytes=image_bytes)
    print(tensor)


tensor([[[[-0.1657, -0.3883, -0.5938,  ..., -0.9192, -0.9192, -0.9192],
          [-0.0287, -0.2342, -0.4226,  ..., -0.8507, -0.8678, -0.8678],
          [ 0.0569, -0.1143, -0.2856,  ..., -0.7822, -0.7993, -0.8164],
          ...,
          [ 1.7694,  1.7352,  1.7523,  ..., -0.5767, -0.4739, -0.3027],
          [ 1.8037,  1.7180,  1.7523,  ..., -0.5424, -0.4568, -0.2684],
          [ 1.7352,  1.7694,  1.7352,  ..., -0.5253, -0.4568, -0.2684]],

         [[ 0.0301, -0.1975, -0.4251,  ..., -0.5301, -0.5301, -0.5301],
          [ 0.2052, -0.0049, -0.2150,  ..., -0.4601, -0.4776, -0.4776],
          [ 0.3102,  0.1176, -0.0749,  ..., -0.3901, -0.4076, -0.4251],
          ...,
          [ 1.7983,  1.7633,  1.7808,  ..., -0.2325, -0.1625, -0.0049],
          [ 1.8508,  1.7283,  1.7633,  ..., -0.2325, -0.1450,  0.0301],
          [ 1.7633,  1.7808,  1.7633,  ..., -0.2500, -0.1800, -0.0224]],

         [[ 0.1999, -0.1138, -0.4101,  ..., -1.0898, -1.0898, -1.0898],
          [ 0.4265,  0.1476, -

In [3]:
#  Now prepare the prediction part:

from torchvision import models

# Make sure to pass `pretrained` as `True` to use the pretrained weights:
model = models.densenet121(pretrained=True)
# Since we are using our model only for inference, switch to `eval` mode:
model.eval()


def get_prediction(image_bytes):
    tensor = transform_image(image_bytes=image_bytes)
    outputs = model.forward(tensor)
    _, y_hat = outputs.max(1)
    return y_hat

/Users/bea/Documents/University/Y2/T2/Artificial-Intelligence/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/bea/Documents/University/Y2/T2/Artificial-Intelligence/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


The issue now is we would like the classification of the image to make sense to humans, and not just be a number.

For that, we need to use the file `imagenet_class_index.json` --> save it somewhere where you know where it is.

In [4]:
import json

imagenet_class_index = json.load(open('imagenet_class_index.json'))

def get_prediction(image_bytes):
    tensor = transform_image(image_bytes=image_bytes)
    outputs = model.forward(tensor)
    _, y_hat = outputs.max(1)
    predicted_idx = str(y_hat.item())
    return imagenet_class_index[predicted_idx]

In [5]:
#  To test this above method using an image that we have:

with open("kitten.jpg", 'rb') as f:
    image_bytes = f.read()
    print(get_prediction(image_bytes=image_bytes))

['n02123159', 'tiger_cat']


Not bad!